# Evaluation A2 — Jurisdiction-OOD AMP Results

This notebook loads the finalized pooled, fold, jurisdiction, shift, and sensitivity artifacts for the frozen A2 design. It does not recompute metrics or figures.

`PURPOSE_REMOVAL_OF_ORGANS` remains a prediction dimension but has zero positive A2 silver-reference support. Its per-label F1 is undefined where appropriate, and the official pooled A2 Macro-F1 uses the 16 supported labels. Jurisdiction rows are descriptive; small-N results must not be ranked as “best” or “worst.”

## 1. Setup and finalized-artifact gate

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, SVG, display


def locate_repo_root() -> Path:
    configured = os.environ.get("SHERLOC_REPO_ROOT")
    starts = [Path(configured).expanduser()] if configured else []
    starts.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in starts:
        if (candidate / "src/experiments/11_evaluate_amp.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate SHERLOC_Case_Analysis. Start Jupyter in the repository "
        "or set SHERLOC_REPO_ROOT."
    )


REPO_ROOT = locate_repo_root()
ANALYSIS_ROOT = REPO_ROOT / "outputs/analysis/evaluation_a"
FIGURE_ROOT = REPO_ROOT / "outputs/figures/evaluation_a"
METRICS_ROOT = REPO_ROOT / "outputs/metrics"


def load_csv(path: Path, required_columns=()) -> pd.DataFrame:
    """Load a finalized artifact without synthesizing missing rows."""
    if not path.is_file():
        display(Markdown(f"> **NOT YET AVAILABLE:** `{path.relative_to(REPO_ROOT)}`"))
        return pd.DataFrame(columns=list(required_columns))
    frame = pd.read_csv(path)
    missing = set(required_columns) - set(frame.columns)
    if missing:
        raise ValueError(f"{path} is missing required columns: {sorted(missing)}")
    return frame


def load_json(path: Path) -> dict:
    if not path.is_file():
        display(Markdown(f"> **NOT YET AVAILABLE:** `{path.relative_to(REPO_ROOT)}`"))
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def show_table(frame: pd.DataFrame, *, empty_message="No finalized rows are available."):
    if frame.empty:
        display(Markdown(f"> **NOT YET AVAILABLE:** {empty_message}"))
    else:
        display(frame)


def show_figure(filename: str):
    """Display a finalized SVG; never recreate a figure in the notebook."""
    path = FIGURE_ROOT / filename
    if not path.is_file():
        display(Markdown(f"> **NOT YET AVAILABLE:** `{path.relative_to(REPO_ROOT)}`"))
        return
    display(SVG(filename=str(path)))


manifest = load_json(METRICS_ROOT / "amp_evaluation_manifest.json")
completion_gate = manifest.get("final_completion_gate", "NOT YET AVAILABLE")
display(Markdown(f"**Canonical Evaluation A completion gate:** `{completion_gate}`"))


In [ ]:
table_names = ['a2_main_comparison.csv', 'amp_family_level_metrics.csv', 'prediction_breadth_summary.csv', 'rare_label_sensitivity.csv', 'a1_to_a2_distribution_shift.csv', 'm3_vs_m4_summary.csv', 'm3_vs_m4_per_label_f1.csv', 'amp_label_display_mapping.csv', 'a2_fold_summary.csv', 'a2_jurisdiction_summary.csv']
figure_names = ['figure_1_a1_vs_a2_core_performance.svg', 'figure_2_cpmr_by_amp_family.svg', 'figure_3_cpmr_vs_contained_recall.svg', 'figure_4_per_label_f1.svg']
artifact_inventory = pd.DataFrame([
    *[
        {"artifact": str((ANALYSIS_ROOT / name).relative_to(REPO_ROOT)),
          "kind": "paper-facing table", "available": (ANALYSIS_ROOT / name).is_file()}
        for name in table_names
    ],
    *[
        {"artifact": str((FIGURE_ROOT / name).relative_to(REPO_ROOT)),
          "kind": "finalized figure", "available": (FIGURE_ROOT / name).is_file()}
        for name in figure_names
    ],
])
display(artifact_inventory)
if not artifact_inventory["available"].all():
    display(Markdown("> **NOT YET AVAILABLE:** one or more finalized artifacts are missing."))


## 2. Canonical pooled A2 comparison

In [ ]:
a2_main = load_csv(ANALYSIS_ROOT / "a2_main_comparison.csv", ("method", "n"))
show_table(a2_main)


## 3. Family-level performance and prediction breadth

In [ ]:
family_metrics = load_csv(
    ANALYSIS_ROOT / "amp_family_level_metrics.csv", ("evaluation", "method", "family")
)
prediction_breadth = load_csv(
    ANALYSIS_ROOT / "prediction_breadth_summary.csv", ("evaluation", "method")
)
show_table(family_metrics.loc[family_metrics["evaluation"].eq("A2")])
show_table(prediction_breadth.loc[prediction_breadth["evaluation"].eq("A2")])


## 4. A1 to A2 distribution shift

Every delta is pooled A2 minus A1. These are descriptive differences; statistical significance was not tested.

In [ ]:
shift = load_csv(
    ANALYSIS_ROOT / "a1_to_a2_distribution_shift.csv", ("method",)
)
show_table(shift)


## 5. Descriptive M4 minus M3 comparison

In [ ]:
m3_m4 = load_csv(ANALYSIS_ROOT / "m3_vs_m4_summary.csv", ("evaluation",))
m3_m4_per_label = load_csv(
    ANALYSIS_ROOT / "m3_vs_m4_per_label_f1.csv", ("evaluation", "label_id")
)
show_table(m3_m4.loc[m3_m4["evaluation"].eq("A2")])
show_table(m3_m4_per_label.loc[m3_m4_per_label["evaluation"].eq("A2")])


## 6. Fold and jurisdiction summaries

In [ ]:
fold_summary = load_csv(
    ANALYSIS_ROOT / "a2_fold_summary.csv", ("method", "fold", "n")
)
jurisdiction_summary = load_csv(
    ANALYSIS_ROOT / "a2_jurisdiction_summary.csv", ("jurisdiction", "method", "n")
)
show_table(fold_summary)
show_table(jurisdiction_summary)
display(Markdown(
    "Per-jurisdiction estimates are descriptive only. Do not rank jurisdictions or overinterpret small N."
))


## 7. Rare-label sensitivity and support rule

In [ ]:
rare_sensitivity = load_csv(
    ANALYSIS_ROOT / "rare_label_sensitivity.csv", ("evaluation", "method")
)
show_table(rare_sensitivity.loc[rare_sensitivity["evaluation"].eq("A2")])
display(Markdown(
    "A2 Organ Removal support is zero. It remains in predictions and micro/set metrics but is already excluded from the official 16-supported-label Macro-F1."
))


## 8. Recorded M3/M4 API execution summary

In [ ]:
a2_api_usage = load_csv(
    METRICS_ROOT / "a2/amp_llm_api_usage.csv", ("method", "scope")
)
show_table(a2_api_usage)


## 9. Four core paper figures

In [ ]:
for figure_name in (
    "figure_1_a1_vs_a2_core_performance.svg",
    "figure_2_cpmr_by_amp_family.svg",
    "figure_3_cpmr_vs_contained_recall.svg",
    "figure_4_per_label_f1.svg",
):
    show_figure(figure_name)

label_display_mapping = load_csv(
    ANALYSIS_ROOT / "amp_label_display_mapping.csv",
    ("ontology_order", "label_id", "family", "display_label", "figure_short_label"),
)
display(Markdown("**Figure 4 short-label mapping to the full frozen ontology:**"))
show_table(label_display_mapping)
